# 07. One-pass self-training

Owner: rdevasia25. Issue #7. Depends on completed #4.

## Goal

Test one fixed pseudo-label pass for C1 and the training-CV-selected C4 word teacher. Compare each treatment with a fresh matched supervised control.

## Setup

A confident teacher assigns temporary labels to eligible unlabeled text. Those labels can be wrong and reinforce the teacher, so use low weight and preserve the control comparison.

In [ ]:
import json, time, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import vstack
from IPython import get_ipython
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
get_ipython().run_line_magic("run", "./02_split_metrics.ipynb")
NAME = "07_pseudo_labels"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=900 notebooks/07_pseudo_labels.ipynb"
CONFIG = {"teacher_C": [1., 4.], "confidence": .95, "pseudo_label_threshold": .5,
    "verified_weight": 1., "pseudo_weight": .25, "passes": 1, "threads": 1,
    "vocabulary_idf": "fit on verified training only; frozen during pseudo-label refit",
    "fit_row_definition": "8000 verified rows; report pseudo and total treatment fit rows separately",
    "vectorizer": {"analyzer": "word", "ngram_range": [1, 2], "lowercase": True,
                   "min_df": 2, "max_features": 50000, "sublinear_tf": True,
                   "norm": "l2", "dtype": "float32", "stop_words": None,
                   "token_pattern": r"(?u)\b\w\w+\b", "strip_accents": None},
    "classifier": {"solver": "liblinear", "max_iter": 2000, "random_state": 42, "tol": 1e-4},
    "bootstrap": {"draws": 2000, "seed": 42, "sampling": "paired class-stratified percentile 95%"},
    "selection": "fixed ablation only; no tuning of confidence or sample weights on validation"}
CONFIG["notebook_source_sha256"] = value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/07_pseudo_labels.ipynb").read_text())["cells"]])
START = time.perf_counter()


## Checks

Use notebook 02 eligibility, which excludes duplicate groups touching validation or test. Check the selected word logistic C against #4 rather than assuming the requested C4 won.

In [ ]:
parts = load_partitions()
train, validation, unlabeled = parts["train"], parts["validation"], parts["unlabeled"]
require(unlabeled.id.tolist() == parts["split"]["eligible_unlabeled_ids"], "Unlabeled eligibility changed")
require(not set(unlabeled.id) & set(validation.id), "Pseudo-label IDs touch validation")
word_pointer = json.loads((ROOT / "artifacts/04_word_comparison/latest.json").read_text())
require(word_pointer["split_sha256"] == parts["split"]["split_sha256"], "Word split differs")
require(file_hash(ROOT / word_pointer["result"]) == word_pointer["result_sha256"], "Word result changed")
word_record = json.loads((ROOT / word_pointer["result"]).read_text())
selected_word = word_pointer["models"]["logistic"]["candidate"]
require(selected_word == {"family": "logistic", "C": 4.0}, "C4 was not selected; update the documented experiment before fitting")
require(CONFIG["vectorizer"] == word_record["config"]["vectorizer"], "Word settings changed")
CONFIG["word_result"] = word_pointer["result"]
CONFIG["word_result_sha256"] = word_pointer["result_sha256"]
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / run_id
private.mkdir(parents=True, exist_ok=False)


## Method

Accept max(p, 1-p) at least .95, then use hard labels at .5. Verified rows have weight 1; accepted pseudo rows have weight .25. Freeze supervised vocabulary and IDF and refit only the classifier once. Fit each control from scratch with the same seed and verified data.

The confidence threshold is a teacher score, not evidence that 95% of accepted labels are correct. We have no true labels for the unlabeled rows.

In [ ]:
def make_teacher(c):
    settings = dict(CONFIG["vectorizer"])
    settings["ngram_range"], settings["dtype"] = tuple(settings["ngram_range"]), np.float32
    return Pipeline([("tfidf", TfidfVectorizer(**settings)),
                     ("classifier", LogisticRegression(C=c, **CONFIG["classifier"]))])

def predictions_for(model, frame):
    return pd.DataFrame({"id": frame.id.to_numpy(), "p_ai": class1_probabilities(model, frame.text)})

def paired_interval(truth, reference, treatment):
    evaluate_predictions(truth, reference); evaluate_predictions(truth, treatment)
    merged = truth[["id", "label"]].merge(reference.rename(columns={"p_ai": "reference"}), on="id", validate="one_to_one")
    merged = merged.merge(treatment.rename(columns={"p_ai": "treatment"}), on="id", validate="one_to_one")
    y, a, b = (merged[k].to_numpy() for k in ["label", "reference", "treatment"])
    groups = [np.flatnonzero(y == label) for label in [0, 1]]
    rng = np.random.default_rng(CONFIG["bootstrap"]["seed"])
    differences = []
    for _ in range(CONFIG["bootstrap"]["draws"]):
        idx = np.concatenate([rng.choice(group, len(group), replace=True) for group in groups])
        differences.append(roc_auc_score(y[idx], b[idx]) - roc_auc_score(y[idx], a[idx]))
    low, high = np.quantile(differences, [.025, .975])
    return {"delta_auroc": float(roc_auc_score(y, b) - roc_auc_score(y, a)),
            "ci_low": float(low), "ci_high": float(high)}

def matrix_probabilities(classifier, matrix):
    classes = list(classifier.classes_)
    require(set(classes) == {0, 1}, "Wrong classifier classes")
    probabilities = np.asarray(classifier.predict_proba(matrix))
    require(probabilities.shape == (matrix.shape[0], 2), "Wrong matrix probability shape")
    require(np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all(), "Invalid matrix probabilities")
    return probabilities[:, classes.index(1)]

def feature_identity(vectorizer):
    return {"vocabulary": value_hash(vectorizer.get_feature_names_out().tolist()),
            "idf": hashlib.sha256(vectorizer.idf_.tobytes()).hexdigest()}


## Run

First train teachers, choose eligible pseudo rows, and fit matched arms. Save acceptance counts and settings before validation. Then evaluate the fixed comparisons once.

In [ ]:
artifacts, acceptance, fitted = [], [], []
try:
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        for c in CONFIG["teacher_C"]:
            teacher = make_teacher(c)
            teacher_start = time.perf_counter()
            teacher.fit(train.text, train.label)
            teacher_seconds = time.perf_counter() - teacher_start
            vectorizer = teacher.named_steps["tfidf"]
            identity_before = feature_identity(vectorizer)
            feature_start = time.perf_counter()
            x_train = vectorizer.transform(train.text)
            x_unlabeled = vectorizer.transform(unlabeled.text)
            pseudo_probability = matrix_probabilities(teacher.named_steps["classifier"], x_unlabeled)
            keep = np.maximum(pseudo_probability, 1 - pseudo_probability) >= CONFIG["confidence"]
            pseudo_y = (pseudo_probability[keep] >= CONFIG["pseudo_label_threshold"]).astype(int)
            feature_seconds = time.perf_counter() - feature_start
            accepted = int(keep.sum())
            control = LogisticRegression(C=c, **CONFIG["classifier"])
            control_start = time.perf_counter()
            control.fit(x_train, train.label, sample_weight=np.ones(len(train)))
            control_seconds = time.perf_counter() - control_start
            treatment = LogisticRegression(C=c, **CONFIG["classifier"])
            x_augmented = vstack([x_train, x_unlabeled[keep]], format="csr")
            y_augmented = np.concatenate([train.label.to_numpy(), pseudo_y])
            weights = np.concatenate([np.ones(len(train)), np.full(accepted, CONFIG["pseudo_weight"])])
            treatment_start = time.perf_counter()
            treatment.fit(x_augmented, y_augmented, sample_weight=weights)
            treatment_seconds = time.perf_counter() - treatment_start
            require(feature_identity(vectorizer) == identity_before, "Pseudo rows changed vocabulary/IDF")
            teacher_control_difference = float(np.max(np.abs(matrix_probabilities(teacher.named_steps["classifier"], x_train) - matrix_probabilities(control, x_train))))
            require(teacher_control_difference <= 1e-10, "Matched supervised control differs from teacher")
            row = {"C": c, "eligible_rows": len(unlabeled), "accepted_rows": accepted,
                "accepted_human": int((pseudo_y == 0).sum()), "accepted_AI": int((pseudo_y == 1).sum()),
                "verified_weighted_mass": float(len(train)), "pseudo_weighted_mass": float(.25 * accepted),
                "total_weighted_mass": float(len(train) + .25 * accepted), "treatment_fit_rows": len(train) + accepted,
                "teacher_fit_seconds": teacher_seconds, "feature_seconds": feature_seconds,
                "control_fit_seconds": control_seconds, "treatment_fit_seconds": treatment_seconds,
                "max_teacher_control_difference": teacher_control_difference, "feature_identity": identity_before}
            acceptance.append(row)
            pseudo_path = private / f"C{c}_accepted_pseudo.csv"
            pd.DataFrame({"id": unlabeled.id.to_numpy()[keep], "pseudo_label": pseudo_y,
                          "teacher_p_ai": pseudo_probability[keep], "sample_weight": .25}).to_csv(pseudo_path, index=False)
            artifacts.append(pseudo_path)
            fitted.append({"C": c, "teacher": teacher,
                "control": Pipeline([("tfidf", vectorizer), ("classifier", control)]),
                "treatment": Pipeline([("tfidf", vectorizer), ("classifier", treatment)])})
    selection_path = ROOT / "results" / NAME / f"{run_id}-pseudo-selection.json"
    save_new_json(selection_path, {"acceptance": acceptance, "config": CONFIG,
        "validation_inspected": False, "utc": datetime.now(timezone.utc).isoformat()})
    artifacts.append(selection_path)
    result_rows, comparisons, saved = [], [], []
    with threadpool_limits(limits=1):
        for arm in fitted:
            c = arm["C"]
            predictions = {}
            model_info = {"C": c}
            for name in ["teacher", "control", "treatment"]:
                model = arm[name]
                evaluation_start = time.perf_counter()
                train_pred, valid_pred = predictions_for(model, train), predictions_for(model, validation)
                tm, vm = evaluate_predictions(train, train_pred), evaluate_predictions(validation, valid_pred)
                evaluation_seconds = time.perf_counter() - evaluation_start
                model_path = private / f"C{c}_{name}.joblib"
                train_path, valid_path = private / f"C{c}_{name}_train.csv", private / f"C{c}_{name}_validation.csv"
                joblib.dump(model, model_path)
                train_pred.to_csv(train_path, index=False); valid_pred.to_csv(valid_path, index=False)
                diff = float(np.max(np.abs(class1_probabilities(joblib.load(model_path), validation.text) - valid_pred.p_ai.to_numpy())))
                require(diff <= 1e-12, "Pseudo experiment reload mismatch")
                artifacts.extend([model_path, train_path, valid_path])
                predictions[name] = valid_pred
                model_info[name] = {"model": str(model_path.relative_to(ROOT)),
                    "train_predictions": str(train_path.relative_to(ROOT)), "validation_predictions": str(valid_path.relative_to(ROOT))}
                result_rows.append({"C": c, "arm": name, "train_auroc": tm["auroc"], "validation_auroc": vm["auroc"],
                    "validation_accuracy_at_0_5": vm["accuracy_at_0_5"], "evaluation_seconds": evaluation_seconds,
                    "max_reload_difference": diff})
            bootstrap_start = time.perf_counter()
            ci = paired_interval(validation, predictions["control"], predictions["treatment"])
            comparisons.append({"C": c, **ci, "bootstrap_seconds": time.perf_counter() - bootstrap_start})
            saved.append(model_info)
except Exception as exc:
    record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation),
        {"validation_auroc": None, "acceptance": acceptance, "arm_results": locals().get("result_rows", [])},
        time.perf_counter() - START, status="failed", error_type=type(exc).__name__)
    raise RuntimeError("Self-training failed; see safe run record.") from None


## Results

Both teacher settings and matched controls remain visible. Bootstrap intervals use 2,000 paired class-stratified draws at seed 42, conditional on these fitted models and reused validation rows. They exclude training variability and cannot verify pseudo-label correctness.

In [ ]:
display(pd.DataFrame(acceptance).drop(columns="feature_identity"))
display(pd.DataFrame(result_rows))
display(pd.DataFrame(comparisons))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for i, c in enumerate(CONFIG["teacher_C"]):
    rows = [r for r in result_rows if r["C"] == c and r["arm"] != "teacher"]
    x = np.arange(len(rows))
    for partition, offset in [("train", -.18), ("validation", .18)]:
        axes[i].bar(x + offset, [r[f"{partition}_auroc"] for r in rows], width=.36, label=partition)
    axes[i].set(xticks=x, xticklabels=[r["arm"] for r in rows], ylim=(0, 1), ylabel="AUROC", title=f"C{c}: one pseudo-label pass")
    axes[i].legend()
fig.tight_layout()
plot_path = private / "self_training.png"
fig.savefig(plot_path, dpi=140); artifacts.append(plot_path)
plt.show()
for row in comparisons:
    print(f"C{row['C']}: treatment-control AUROC {row['delta_auroc']:+.6f}, 95% interval [{row['ci_low']:+.6f}, {row['ci_high']:+.6f}]")


## Takeaway

- Tried one fixed pseudo-label pass with frozen supervised features and quarter-weight pseudo rows.
- The tables show actual accepted counts, weighted mass, and treatment-control changes. Keep negligible or negative outcomes.
- Next: normalized features and teacher transfer in #15 after #13. Do not retune confidence or weights on this validation set.

## Save

Commit measured aggregates only. Teacher probabilities, accepted pseudo-label rows, models, and aligned predictions remain under ignored artifacts. This is a fixed ablation, with no automatic model adoption or upload.

In [ ]:
metrics = {"validation_auroc": None, "acceptance": acceptance, "arm_results": result_rows,
    "paired_treatment_control": comparisons, "selection": "no automatic adoption; both fixed comparisons retained"}
CONFIG["extra_versions"] = {name: importlib.metadata.version(name) for name in ["joblib", "matplotlib", "threadpoolctl"]}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), metrics, time.perf_counter() - START, artifacts=artifacts)
pointer = {"result": str(record_path.relative_to(ROOT)), "result_sha256": file_hash(record_path),
    "split_sha256": parts["split"]["split_sha256"], "models": saved,
    "artifacts": {str(p.relative_to(ROOT)): file_hash(p) for p in artifacts}}
pointer_path = ROOT / "artifacts" / NAME / "latest.json"
temporary = pointer_path.with_suffix(".tmp")
temporary.write_text(json.dumps(pointer, indent=2, sort_keys=True) + "\n"); temporary.replace(pointer_path)
print("Saved both self-training comparisons and private artifacts.")
